# EYES-DEFY-ANEMIA -- EfficientNet-B0 (forniceal_palpebral) -- 5-Fold Cross-Validation

Dedicated follow-up pipeline for the single best batch-1 performer (val F1=0.933). Fresh random head init per fold (no warm-start), hyperparameters locked from batch-1 trial #9 (no Optuna), gradient clipping, `ReduceLROnPlateau` + `EarlyStopping`, batch size 32, 150-epoch ceiling, unchanged augmentation baseline (flip + rotate only). The 33-patient test set is permanently excluded from all 5 folds. Full rationale in `classification/.project_memory/02_current_status.md`.

Each fold runs as its own cell (`--fold N`), not one long call, specifically so a mid-run Kaggle session interruption only loses the fold in progress -- the same incremental-safety pattern as `classification-final-fixed.ipynb` (batch 1) and `classification-batch2.ipynb`.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia
!git pull  # belt-and-suspenders after a fresh clone -- see memory notes for why this is a safe no-op normally

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 435, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 435 (delta 35), reused 100 (delta 29), pack-reused 319 (from 1)
Receiving objects: 100% (435/435), 61.37 MiB | 35.54 MiB/s, done.
Resolving deltas: 100% (198/198), done.
/kaggle/working/eyes-defy-anemia
Already up to date.


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [5]:
import shutil
from pathlib import Path

SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


In [6]:
import sys

sys.path.insert(0, "classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv")
from cv_dataset import load_cv_pool, build_folds, N_FOLDS

pool = load_cv_pool()
print(f"CV pool: {len(pool)} patients (expect 178 -- 33-patient test set excluded)")
assert len(pool) == 178, f"expected 178, got {len(pool)}"

folds = build_folds(pool, n_folds=N_FOLDS)
print(f"Built {len(folds)} folds:")
for i, (train_df, val_df) in enumerate(folds, start=1):
    print(f"  Fold {i}: train={len(train_df)} val={len(val_df)}")

CV pool: 178 patients (expect 178 -- 33-patient test set excluded)
Built 5 folds:
  Fold 1: train=142 val=36
  Fold 2: train=142 val=36
  Fold 3: train=142 val=36
  Fold 4: train=143 val=35
  Fold 5: train=143 val=35


## sync_outputs() -- consolidate + zip after every fold

Same design as batch 1/2: copies `classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/outputs/{checkpoints,logs,plots}/` into a top-level `/kaggle/working/outputs/` and zips it to `/kaggle/working/efficientnet_b0_cv_results.zip`. Called after every one of the 6 training cells below (5 folds + 1 aggregation step), not just at the end.

In [7]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate this pipeline's own outputs/{checkpoints,logs,plots}/
    into a top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/efficientnet_b0_cv_results.zip. Called after every
    fold (and the final aggregation step) so a mid-run interruption --
    a real risk over 5 folds x up to 150 epochs, by far the longest run
    in this project -- still leaves a complete, downloadable snapshot
    of whatever finished."""
    src_dir = Path("classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/outputs")
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        sub_src = src_dir / sub
        if sub_src.exists():
            shutil.copytree(sub_src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive(
        "/kaggle/working/efficientnet_b0_cv_results", "zip", root_dir=str(results_dir)
    )
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/efficientnet_b0_cv_results.zip


## Training -- 5 folds, then aggregate

Each cell is a separate `!python` call to `run_cv_training.py`, one fold at a time (`--fold N`), then a final `--aggregate` pass that reads all 5 folds' saved history and computes the cross-fold summary -- no re-training. A failed cell does not halt "Run All" (same IPython behavior as batch 1/2) -- check each cell's own output, or the saved `.../outputs/logs/efficientnet_b0_forniceal_palpebral_cv_fold*_history.json` files, after this finishes.

In [8]:
# Fold 1 / 5
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 1
sync_outputs()

Using device: cuda
Model: efficientnet_b0_forniceal_palpebral_cv
Locked hyperparameters: lr=0.0008200518402245837, weight_decay=1.9634341572933354e-06, dropout=0.2 (from Batch 1 trial #9, no Optuna)
batch_size=32, max_epochs=150, grad_clip_max_norm=1.0, scheduler=ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-06), early_stopping_patience=15

CV pool: 178 patients (33-patient test set excluded, never loaded)

Fold 1/5 -- train=142 val=36
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 139MB/s]
[Fold 1] New best val_f1=0.2857 -> saved efficientnet_b0_forniceal_palpebral_cv_fold1_best.pth
[Fold 1] Epoch   1/150 - train_loss=0.7791 val_loss=0.7987 val_f1=0.2857 lr=8.20e-04
[Fold 1] Epoch   2/150 - train_loss=0.7347 val_loss=0.7859 val_f1=0.1176 lr=8.20e-04
[Fold 1] New best val_f1=0.3000 -> saved 

In [9]:
# Fold 2 / 5
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 2
sync_outputs()

Using device: cuda
Model: efficientnet_b0_forniceal_palpebral_cv
Locked hyperparameters: lr=0.0008200518402245837, weight_decay=1.9634341572933354e-06, dropout=0.2 (from Batch 1 trial #9, no Optuna)
batch_size=32, max_epochs=150, grad_clip_max_norm=1.0, scheduler=ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-06), early_stopping_patience=15

CV pool: 178 patients (33-patient test set excluded, never loaded)

Fold 2/5 -- train=142 val=36
[Fold 2] New best val_f1=0.4000 -> saved efficientnet_b0_forniceal_palpebral_cv_fold2_best.pth
[Fold 2] Epoch   1/150 - train_loss=0.7741 val_loss=0.7959 val_f1=0.4000 lr=8.20e-04
[Fold 2] Epoch   2/150 - train_loss=0.7412 val_loss=0.7953 val_f1=0.1176 lr=8.20e-04
[Fold 2] Epoch   3/150 - train_loss=0.7036 val_loss=0.7886 val_f1=0.1176 lr=8.20e-04
[Fold 2] Epoch   4/150 - train_loss=0.6830 val_loss=0.7688 val_f1=0.1176 lr=8.20e-04
[Fold 2] Epoch   5/150 - train_loss=0.6586 val_loss=0.7273 val_f1=0.3158 lr=8.20e-04
[Fold 2] New best val_f1=0.5833 ->

In [10]:
# Fold 3 / 5
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 3
sync_outputs()

Using device: cuda
Model: efficientnet_b0_forniceal_palpebral_cv
Locked hyperparameters: lr=0.0008200518402245837, weight_decay=1.9634341572933354e-06, dropout=0.2 (from Batch 1 trial #9, no Optuna)
batch_size=32, max_epochs=150, grad_clip_max_norm=1.0, scheduler=ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-06), early_stopping_patience=15

CV pool: 178 patients (33-patient test set excluded, never loaded)

Fold 3/5 -- train=142 val=36
[Fold 3] New best val_f1=0.1250 -> saved efficientnet_b0_forniceal_palpebral_cv_fold3_best.pth
[Fold 3] Epoch   1/150 - train_loss=0.7633 val_loss=0.7711 val_f1=0.1250 lr=8.20e-04
[Fold 3] New best val_f1=0.2353 -> saved efficientnet_b0_forniceal_palpebral_cv_fold3_best.pth
[Fold 3] Epoch   2/150 - train_loss=0.7403 val_loss=0.7655 val_f1=0.2353 lr=8.20e-04
[Fold 3] New best val_f1=0.4211 -> saved efficientnet_b0_forniceal_palpebral_cv_fold3_best.pth
[Fold 3] Epoch   3/150 - train_loss=0.7078 val_loss=0.7355 val_f1=0.4211 lr=8.20e-04
[Fold 3] New b

In [11]:
# Fold 4 / 5
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 4
sync_outputs()

Using device: cuda
Model: efficientnet_b0_forniceal_palpebral_cv
Locked hyperparameters: lr=0.0008200518402245837, weight_decay=1.9634341572933354e-06, dropout=0.2 (from Batch 1 trial #9, no Optuna)
batch_size=32, max_epochs=150, grad_clip_max_norm=1.0, scheduler=ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-06), early_stopping_patience=15

CV pool: 178 patients (33-patient test set excluded, never loaded)

Fold 4/5 -- train=143 val=35
[Fold 4] New best val_f1=0.4348 -> saved efficientnet_b0_forniceal_palpebral_cv_fold4_best.pth
[Fold 4] Epoch   1/150 - train_loss=0.7837 val_loss=0.7927 val_f1=0.4348 lr=8.20e-04
[Fold 4] New best val_f1=0.4848 -> saved efficientnet_b0_forniceal_palpebral_cv_fold4_best.pth
[Fold 4] Epoch   2/150 - train_loss=0.7363 val_loss=0.7928 val_f1=0.4848 lr=8.20e-04
[Fold 4] New best val_f1=0.6000 -> saved efficientnet_b0_forniceal_palpebral_cv_fold4_best.pth
[Fold 4] Epoch   3/150 - train_loss=0.7043 val_loss=0.7784 val_f1=0.6000 lr=8.20e-04
[Fold 4] Epoch

In [12]:
# Fold 5 / 5
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --fold 5
sync_outputs()

Using device: cuda
Model: efficientnet_b0_forniceal_palpebral_cv
Locked hyperparameters: lr=0.0008200518402245837, weight_decay=1.9634341572933354e-06, dropout=0.2 (from Batch 1 trial #9, no Optuna)
batch_size=32, max_epochs=150, grad_clip_max_norm=1.0, scheduler=ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-06), early_stopping_patience=15

CV pool: 178 patients (33-patient test set excluded, never loaded)

Fold 5/5 -- train=143 val=35
[Fold 5] New best val_f1=0.1176 -> saved efficientnet_b0_forniceal_palpebral_cv_fold5_best.pth
[Fold 5] Epoch   1/150 - train_loss=0.7710 val_loss=0.7687 val_f1=0.1176 lr=8.20e-04
[Fold 5] New best val_f1=0.4211 -> saved efficientnet_b0_forniceal_palpebral_cv_fold5_best.pth
[Fold 5] Epoch   2/150 - train_loss=0.7371 val_loss=0.7488 val_f1=0.4211 lr=8.20e-04
[Fold 5] New best val_f1=0.4762 -> saved efficientnet_b0_forniceal_palpebral_cv_fold5_best.pth
[Fold 5] Epoch   3/150 - train_loss=0.6962 val_loss=0.7256 val_f1=0.4762 lr=8.20e-04
[Fold 5] New b

In [13]:
# Aggregate -- reads all 5 folds' saved history, computes the cross-fold summary. No training.
!python classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/run_cv_training.py --aggregate
sync_outputs()


5-Fold CV Summary -- efficientnet_b0_forniceal_palpebral_cv
  overall : F1=0.8667+/-0.0526  BalAcc=0.8840+/-0.0529  AUC=0.8763+/-0.0663
  India   : F1=0.8836+/-0.0246  BalAcc=0.6750+/-0.0548  AUC=0.6392+/-0.0568
  Italy   : F1=0.8305+/-0.1708  BalAcc=0.9108+/-0.1023  AUC=0.9221+/-0.0985

Saved cross-fold summary to /kaggle/working/eyes-defy-anemia/classification/scripts/efficientnet_b0_forniceal_5fold_cv/outputs/logs/efficientnet_b0_forniceal_palpebral_cv_cv_summary.json
[sync_outputs] 36 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/efficientnet_b0_cv_results.zip


## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever folds completed, plus `efficientnet_b0_forniceal_palpebral_cv_cv_summary.json` if the aggregate step ran) and zipped to `/kaggle/working/efficientnet_b0_cv_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip directly from there, or browse the folder for individual files.

In [14]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/efficientnet_b0_cv_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/efficientnet_b0_forniceal_palpebral_cv_fold1_best.pth  (16.35 MB)
  checkpoints/efficientnet_b0_forniceal_palpebral_cv_fold2_best.pth  (16.35 MB)
  checkpoints/efficientnet_b0_forniceal_palpebral_cv_fold3_best.pth  (16.35 MB)
  checkpoints/efficientnet_b0_forniceal_palpebral_cv_fold4_best.pth  (16.35 MB)
  checkpoints/efficientnet_b0_forniceal_palpebral_cv_fold5_best.pth  (16.35 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_cv_summary.json  (0.02 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_fold1_history.json  (0.01 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_fold2_history.json  (0.01 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_fold3_history.json  (0.01 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_fold4_history.json  (0.01 MB)
  logs/efficientnet_b0_forniceal_palpebral_cv_fold5_history.json  (0.01 MB)
  plots/efficientnet_b0_forniceal_palpebral_cv_fold1_confusion_matrices.png  (0.05 MB)
  plots/efficient